In [1]:
import cv2
import os
import numpy as np
import shutil
from pathlib import Path
from sklearn.model_selection import train_test_split

---
---
---
---

In [2]:

# ============================================================
# 1. الإعدادات التقنية (Global Configuration)
# ============================================================
RAW_DATA_PATH = Path("dataset")      # المجلد الذي يحتوي على فيديوهات normal و cheating
OUTPUT_BASE_PATH = Path("processed_data")

IMG_SIZE = (224, 224)   # المقاس الموحد لجميع الموديلات
SAMPLE_FPS = 6          # عدد الإطارات المستخرجة من كل ثانية (لضمان التقاط الحركة)
SEQ_LENGTH = 16         # عدد الإطارات في كل Sequence
STRIDE = 8              # التداخل بين السيكوينس لزيادة حجم البيانات

In [3]:

# ============================================================
# 2. وظائف المعالجة (Utility Functions)
# ============================================================

def resize_with_padding(frame, target_size=IMG_SIZE):
    """تغيير الحجم مع الحفاظ على الأبعاد وإضافة حواف سوداء"""
    h, w = frame.shape[:2]
    ratio = min(target_size[0]/h, target_size[1]/w)
    new_size = (int(w * ratio), int(h * ratio))
    resized = cv2.resize(frame, new_size)

    canvas = np.zeros((target_size[0], target_size[1], 3), dtype=np.uint8)
    y_offset = (target_size[0] - new_size[1]) // 2
    x_offset = (target_size[1] - new_size[0]) // 2
    canvas[y_offset:y_offset+new_size[1], x_offset:x_offset+new_size[0]] = resized
    return canvas

def extract_clean_frames(video_path):
    """استخراج إطارات الفيديو ومعالجتها تقنياً"""
    cap = cv2.VideoCapture(str(video_path))
    actual_fps = cap.get(cv2.CAP_PROP_FPS)
    hop = max(1, int(actual_fps / SAMPLE_FPS))

    frames = []
    count = 0
    while True:
        ret, frame = cap.read()
        if not ret: break
        if count % hop == 0:
            processed = resize_with_padding(frame)
            frames.append(processed)
        count += 1
    cap.release()
    return frames


In [4]:

# ============================================================
# 3. خط الإنتاج الرئيسي (The Pipeline)
# ============================================================

def run_master_preprocessing():
    # إنشاء المجلدات الفرعية تلقائياً
    for split in ['train', 'val', 'test']:
        for target in ['frames', 'sequences']:
            for cls in ['normal', 'cheating']:
                (OUTPUT_BASE_PATH / target / split / cls).mkdir(parents=True, exist_ok=True)

    for cls in ['normal', 'cheating']:
        vids = list((RAW_DATA_PATH / cls).glob("*.mp4"))
        if not vids: continue

        # تقسيم الفيديوهات (Video-level Split) لمنع تسريب البيانات
        train_v, temp_v = train_test_split(vids, test_size=0.3, random_state=42)
        val_v, test_v = train_test_split(temp_v, test_size=0.5, random_state=42)

        splits = {'train': train_v, 'val': val_v, 'test': test_v}

        for split_name, video_list in splits.items():
            print(f"Current Task: {split_name} | Class: {cls} | Count: {len(video_list)}")

            for v_path in video_list:
                frames = extract_clean_frames(v_path)

                # أ- حفظ الإطارات المنفردة (لأعضاء الـ CNN & ViT)
                for i, frame in enumerate(frames):
                    frame_filename = f"{v_path.stem}_f{i:04d}.jpg"
                    save_path = OUTPUT_BASE_PATH / 'frames' / split_name / cls / frame_filename
                    cv2.imwrite(str(save_path), frame)

                # ب- تكوين وحفظ السلاسل الزمنية (لأعضاء الـ Sequences)
                for i in range(0, len(frames) - SEQ_LENGTH + 1, STRIDE):
                    seq = frames[i : i + SEQ_LENGTH]
                    seq_array = np.array(seq, dtype=np.uint8) # توفير مساحة الهارد ديسك
                    seq_filename = f"{v_path.stem}_s{i:03d}.npy"
                    save_path = OUTPUT_BASE_PATH / 'sequences' / split_name / cls / seq_filename
                    np.save(str(save_path), seq_array)

    print("\n✅ All data is cleaned, organized, and ready for the team!")


In [5]:
if __name__ == "__main__":
    run_master_preprocessing()

Current Task: train | Class: normal | Count: 11
Current Task: val | Class: normal | Count: 3
Current Task: test | Class: normal | Count: 3
Current Task: train | Class: cheating | Count: 78
Current Task: val | Class: cheating | Count: 17
Current Task: test | Class: cheating | Count: 17

✅ All data is cleaned, organized, and ready for the team!
